In [1]:
from gplearn.genetic import SymbolicTransformer
from gplearn.functions import make_function
from sklearn.utils.random import check_random_state

import numpy as np
import array
import pandas
import matplotlib
import math   # 导入 math 模块
import matplotlib.pyplot as plt
import pandas as pd
from pandas import Series, DataFrame
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from itertools import combinations
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
OL= LinearRegression(normalize=False)

plt.rcParams['savefig.dpi'] = 100 #图片像素
plt.rcParams['figure.dpi'] = 100#分辨率

data=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 4\select by pearson from initial-MA-clipped - ranked.csv")

In [2]:
F=np.array(data)
p=data.ev_ratio
#
kfold=8#测试轮数
n=30#每折数据包含的数据条数
#
p_train=np.array(p[0:n*kfold,])
F_train=F[0:n*kfold,3:67]

p_test=np.array(p[n*(kfold+1)-n:n*(kfold+1),])
F_test=F[n*(kfold+1)-n:n*(kfold+1),3:67]

In [3]:
crossover=[]
parsimony=[]
for i in range(25,85,2):
    crossover.append(i*0.01)

for i in range(5,55,10):
    parsimony.append(i*0.00002)
    
crossover_2=[]
subtree_mutation=[]
hoist_mutation=[]

for k in range(len(crossover)):
    up=math.floor((1-crossover[k])*100/3)
    low=math.floor((0.92-crossover[k])*100/3)
    for i in range(low,up,15):
        subtree_mutation.append(i*0.01)
        hoist_mutation.append(i*0.01)
        crossover_2.append(crossover[k])
        
point_mutation=[]
point_mutation=np.array(1-np.array(crossover_2)-np.array(subtree_mutation)-np.array(hoist_mutation))
tot=np.c_[crossover_2,subtree_mutation,hoist_mutation,point_mutation]

In [8]:
function_set = [ 'add','sub','mul', 'div', 'sqrt', 'log','abs','neg', 'inv','sin','cos','tan']

j=len(parsimony)

gen=35
#len(tot)
for m in range(0,j):
    for i in range(len(tot)):
        
        gp = SymbolicTransformer(population_size=10500,tournament_size=20,
                                 metric='pearson',function_set = function_set,init_depth=(4, 15),
                                 generations=gen, stopping_criteria=0.9,
                                 hall_of_fame=500, n_components=1,
                                 p_crossover=tot[i][0],p_subtree_mutation=tot[i][1],
                                 p_hoist_mutation=tot[i][2],p_point_mutation=tot[i][3],
                                 max_samples=0.9, verbose=1,n_jobs=3,
                                 parsimony_coefficient=parsimony[m], random_state=0)

        gp.fit(F_train,(p_train.reshape(-1,1)-1)*100)
        
        for program in gp:
            print(program)
            print(program.length_)
            print(program.raw_fitness_)
        
        gp_features = gp.transform(F_train)
        OL.fit(gp_features, (p_train.reshape(-1,1)-1)*100)
        p_trpre=OL.predict(gp_features)
        r_2tr=r2_score((p_train.reshape(-1,1)-1)*100, p_trpre)
        print(r_2tr)
        
        gp_features_te = gp.transform(F_test)
        OL.fit(gp_features_te, (p_test.reshape(-1,1)-1)*100)
        p_tepre=OL.predict(gp_features_te)
        r_2te=r2_score((p_test.reshape(-1,1)-1)*100, p_tepre)
        print(r_2te)

D:\Python\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    49.76          0.11117        2         0.679116         0.221145     24.38m
   1     8.17         0.315907        2         0.689435         0.330082     11.38m
   2     4.48         0.454783        7           0.7171         0.370699     11.05m
   3     3.74         0.505815        8           0.7305         0.164963     10.15m
   4     4.22         0.518675       12         0.759636         0.498907     11.08m
   5     6.79         0.530479       11         0.773155         0.529518     11.11m
   6     9.03         0.557048       11         0.783621         0.256247      9.70m
   7    10.91         0.593457       11         0.791977         0.484937      9.26m
   8    11.38         0.608001       13          0.79286         0.450585  

D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(


0.6382695496312336
0.011523350471695282
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left


D:\Python\lib\site-packages\sklearn\linear_model\_base.py:148: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2. Please leave the normalize parameter to its default value to silence this warning. The default behavior of this estimator is to not do any normalization. If normalization is needed please use sklearn.preprocessing.StandardScaler instead.
  warnings.warn(
D:\Python\lib\site-packages\sklearn\utils\validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


   0    49.76          0.11117        2         0.679116         0.221145     16.62m
   1     8.09          0.31764        2         0.689435         0.330082     12.35m
   2     4.46         0.458477        7           0.7171         0.370699     10.25m
   3     3.75         0.510832        8           0.7305         0.164963      9.44m
   4     4.29         0.523418       12         0.759636         0.498907      9.26m
   5     6.95         0.531638       11         0.771955         0.297411      9.09m
   6     9.20         0.560301       11         0.783621         0.256247      8.96m
   7    11.13         0.595651       11         0.791977         0.484937      8.79m
   8    11.46         0.610522       14          0.79286         0.450585      8.55m
   9    11.68         0.616453       11         0.792808         0.254564      8.21m
  10    11.56         0.615522       12         0.798695        0.0675609      9.04m
  11    11.39         0.614632       12         0.799973         

KeyboardInterrupt: 